# Starter Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [12]:
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.11.0+cu130
CUDA available: True


In [14]:
# Core training config
from pathlib import Path

default_output_dir = (
    "/kaggle/working/qwen3b_svg_lora_v2"
    if Path("/kaggle/working").exists()
    else str(Path("./artifacts/qwen3b_svg_lora_v2").resolve())
)

CONFIG = {
    "model_name": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    "max_seq_length": 1536,  # covers 88% of training SVGs, fits in 9.6 GiB VRAM
    "lora_r": 16,            
    "lora_alpha": 32,        
    "learning_rate": 2e-4,
    "num_train_epochs": 2,   
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 50,
    "eval_steps": 250,
    "save_steps": 250,
    "max_train_samples_per_source": 50000,
    "eval_size": 0.02,
    "output_dir": default_output_dir,
}

print(f"Using output_dir: {CONFIG['output_dir']}")
CONFIG

Using output_dir: /home/anthonylamelas/Coding/Deep_Learning_Midterm_1/artifacts/qwen3b_svg_lora_v2


{'model_name': 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
 'max_seq_length': 1536,
 'lora_r': 16,
 'lora_alpha': 32,
 'learning_rate': 0.0002,
 'num_train_epochs': 2,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 16,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 50,
 'eval_steps': 250,
 'save_steps': 250,
 'max_train_samples_per_source': 50000,
 'eval_size': 0.02,
 'output_dir': '/home/anthonylamelas/Coding/Deep_Learning_Midterm_1/artifacts/qwen3b_svg_lora_v2'}

In [15]:
# Competition training source (VS Code + Colab extension friendly)
from pathlib import Path

IS_COLAB = False
try:
    from google.colab import drive
    IS_COLAB = True
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

if IS_COLAB:
    DATA_DIR = Path('/content/drive/MyDrive/DL_M1')
    TRAIN_CSV_PATH = str(DATA_DIR / 'train.csv')
    TEST_CSV_PATH = str(DATA_DIR / 'test.csv')
else:
    TRAIN_CSV_PATH = 'data/train.csv'
    TEST_CSV_PATH = 'data/test.csv'

if not Path(TRAIN_CSV_PATH).exists():
    raise FileNotFoundError(f'Train file not found: {TRAIN_CSV_PATH}')
if not Path(TEST_CSV_PATH).exists():
    raise FileNotFoundError(f'Test file not found: {TEST_CSV_PATH}')

print(f'Using train file: {TRAIN_CSV_PATH}')
print(f'Using test file: {TEST_CSV_PATH}')

Using train file: data/train.csv
Using test file: data/test.csv


In [16]:
train_df = pd.read_csv(TRAIN_CSV_PATH)
required_cols = {"prompt", "svg"}
missing = required_cols - set(train_df.columns)
if missing:
    raise ValueError(f"Missing required columns in {TRAIN_CSV_PATH}: {sorted(missing)}")

train_df = train_df.dropna(subset=["prompt", "svg"]).copy()
train_df["prompt"] = train_df["prompt"].astype(str).str.strip()
train_df["svg"] = train_df["svg"].astype(str).str.strip()
train_df = train_df[(train_df["prompt"] != "") & (train_df["svg"].str.lower().str.startswith("<svg"))]

if CONFIG["max_train_samples_per_source"] and len(train_df) > CONFIG["max_train_samples_per_source"]:
    train_df = train_df.sample(CONFIG["max_train_samples_per_source"], random_state=SEED)

if len(train_df) == 0:
    raise RuntimeError("No usable rows found in train.csv after filtering.")

train_raw = Dataset.from_pandas(train_df[["prompt", "svg"]], preserve_index=False)
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]

Train rows: 49000
Eval rows: 1000


{'prompt': 'A black bathtub icon with a faucet, isolated on a white background.',
 'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="" fill-opacity="1.0"  filling="0" d="M48.671878814697266 92.65625 L48.671878814697266 41.40625 L60.2734375 41.40625 L72.8125 53.90625 L83.1640625 43.5546875 L66.3671875 26.7578125 L34.0234375 26.7578125 L34.0234375 92.65625 L26.8359375 92.65625 L45.1171875 165.8984375 L56.015621185302734 165.8984375 L56.015621185302734 173.2421875 L70.6640625 173.2421875 L70.6640625 165.8984375 L129.21875 165.8984375 L129.21875 173.2421875 L143.8671875 173.2421875 L143.8671875 165.8984375 L154.8828125 165.8984375 L173.1640625 92.65625 L48.671878814697266 92.65625 Z"></path></svg>'}

In [17]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Prefer simple shapes (rect, circle, polygon, line, path) with minimal coordinates. "
    "Return only SVG code with a single root <svg> element."
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

Map: 100%|██████████| 1000/1000 [00:00<00:00, 17396.82 examples/s]

<|im_start|>system
You generate compact, valid SVG markup from user requests. Prefer simple shapes (rect, circle, polygon, line, path) with minimal coordinates. Return only SVG code with a single root <svg> element.<|im_end|>
<|im_start|>user
A black bathtub icon with a faucet, isolated on a white background.<|im_end|>
<|im_start|>assistant
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 


In [18]:
from pathlib import Path
import gc

# Free any previously loaded model before loading a new one
if "model" in dir():
    del model
if "_base" in dir():
    del _base
gc.collect()
torch.cuda.empty_cache()

trained_model_dir = "artifacts/qwen3b_svg_lora_v2"
_trained_model_path = Path(trained_model_dir).resolve()

# Check for a fully-saved adapter 
if (_trained_model_path / "adapter_config.json").exists():
    load_dir = str(_trained_model_path)
    load_mode = "inference"
    print(f"Final adapter found — loading for inference: {load_dir}")
elif _trained_model_path.exists():
    checkpoint_dirs = sorted(
        [p for p in _trained_model_path.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
    )
    if checkpoint_dirs:
        load_dir = str(checkpoint_dirs[-1])
        load_mode = "train"
        print(f"Checkpoint found, training in progress — loading via Unsloth: {load_dir}")
    else:
        load_dir = None
        load_mode = "train"
        print("Output dir exists but has no checkpoints yet — loading for training from scratch.")
else:
    load_dir = None
    load_mode = "train"
    print(f"No saved model found — loading for training from scratch.")

if load_mode == "inference":
    # Use vanilla HuggingFace + PEFT for inference — Unsloth patches the RoPE kernel at load
    # time with a training-only implementation that breaks KV-cache decoding (shape mismatch).
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    print(f"Loading for inference via HF + PEFT (no Unsloth RoPE patch): {load_dir}")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    _base = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-3B-Instruct",
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(_base, load_dir)
    tokenizer = AutoTokenizer.from_pretrained(load_dir, trust_remote_code=True)
    tokenizer.padding_side = "left"
    model.eval()
    print(f"Loaded LoRA adapter from: {load_dir}")

else:
    from unsloth import FastLanguageModel

    model_source = CONFIG["model_name"]
    print(f"Loading with Unsloth for training: {model_source}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_source,
        max_seq_length=CONFIG["max_seq_length"],
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        lora_dropout=0,
        bias="none",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )
    if load_dir:
        from peft import set_peft_model_state_dict
        import safetensors.torch as sf
        adapter_weights = sf.load_file(f"{load_dir}/adapter_model.safetensors")
        set_peft_model_state_dict(model, adapter_weights)
        print(f"Loaded LoRA weights from checkpoint: {load_dir}")


Final adapter found — loading for inference: /home/anthonylamelas/Coding/Deep_Learning_Midterm_1/artifacts/qwen3b_svg_lora_v2
Loading for inference via HF + PEFT (no Unsloth RoPE patch): /home/anthonylamelas/Coding/Deep_Learning_Midterm_1/artifacts/qwen3b_svg_lora_v2


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]/home/anthonylamelas/Coding/Deep_Learning_Midterm_1/venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 434/434 [00:04<00:00, 97.56it/s] 


Loaded LoRA adapter from: /home/anthonylamelas/Coding/Deep_Learning_Midterm_1/artifacts/qwen3b_svg_lora_v2


In [19]:
# Training complete — uncomment to retrain
# from pathlib import Path
# import subprocess
# import sysconfig
#
# from trl import SFTTrainer, SFTConfig
#
# python_include_dir = Path(sysconfig.get_paths().get("include", ""))
# python_h = python_include_dir / "Python.h"
# if not python_h.exists():
#     raise EnvironmentError(
#         "Missing Python development headers (Python.h). "
#         f"Expected at: {python_h}. "
#         "Install system package python3.12-dev (or python3-dev), restart the kernel, and rerun from Cell 4."
#     )
#
# if not torch.cuda.is_available():
#     raise EnvironmentError(
#         "CUDA GPU is not available in this kernel, but Unsloth LoRA training requires CUDA/Triton."
#     )
#
# training_args = SFTConfig(
#     output_dir=CONFIG["output_dir"],
#     num_train_epochs=CONFIG["num_train_epochs"],
#     per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
#     gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
#     learning_rate=CONFIG["learning_rate"],
#     warmup_steps=int(CONFIG.get("warmup_steps", 0)) or None,
#     weight_decay=CONFIG["weight_decay"],
#     fp16=not torch.cuda.is_bf16_supported(),
#     bf16=torch.cuda.is_bf16_supported(),
#     logging_steps=CONFIG["logging_steps"],
#     eval_strategy="steps",
#     eval_steps=CONFIG["eval_steps"],
#     save_steps=CONFIG["save_steps"],
#     save_total_limit=3,
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     greater_is_better=False,
#     report_to="none",
#     optim="paged_adamw_8bit",
#     lr_scheduler_type="cosine",
#     seed=SEED,
#     dataset_text_field="text",
#     max_length=CONFIG["max_seq_length"],
#     packing=True,
# )
#
# trainer = SFTTrainer(
#     model=model,
#     processing_class=tokenizer,
#     train_dataset=train_text,
#     eval_dataset=eval_text,
#     args=training_args,
# )
#
# checkpoint_dirs = sorted(
#     [p for p in Path(CONFIG["output_dir"]).glob("checkpoint-*") if p.is_dir()],
#     key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
# )
# resume_checkpoint = str(checkpoint_dirs[-1]) if checkpoint_dirs else None
#
# if resume_checkpoint:
#     for stale in ["optimizer.pt", "scheduler.pt"]:
#         stale_path = Path(resume_checkpoint) / stale
#         if stale_path.exists():
#             stale_path.unlink()
#             print(f"Dropped stale optimizer state: {stale_path.name}")
#     print(f"Resuming from checkpoint: {resume_checkpoint}")
# else:
#     print("No checkpoint found. Starting training from scratch.")
#
# try:
#     if resume_checkpoint:
#         train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
#     else:
#         train_result = trainer.train()
# except subprocess.CalledProcessError as exc:
#     raise RuntimeError(
#         "Triton CUDA build failed during training. "
#         "Most common local fix: install Python dev headers (python3.12-dev) and ensure NVIDIA driver libraries are visible, "
#         "then restart kernel and rerun from Cell 4. "
#         f"Original command: {exc.cmd}"
#     ) from exc
#
# train_result


In [20]:
# Training complete — uncomment to save model after retraining
# os.makedirs(CONFIG["output_dir"], exist_ok=True)
# trainer.save_model(CONFIG["output_dir"])
# tokenizer.save_pretrained(CONFIG["output_dir"])
# print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")


In [21]:
model.eval()
torch.cuda.empty_cache()

# Left-padding required for correct batched causal-LM generation
tokenizer.padding_side = "left"

# Define here so this cell is self-contained for inference-only runs
# (also defined in the training data cell — both values are identical)
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Prefer simple shapes (rect, circle, polygon, line, path) with minimal coordinates. "
    "Return only SVG code with a single root <svg> element."
)

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)
SVG_OPEN_RE = re.compile(r"<svg[\s\S]*", flags=re.IGNORECASE)

# Matches bare & not already part of a valid XML entity
_BARE_AMP_RE = re.compile(r'&(?![a-zA-Z][a-zA-Z0-9]*;|#[0-9]+;|#x[0-9a-fA-F]+;)')

# HTML-only tags that are invalid in SVG (self-closing and paired variants)
_HTML_TAG_RE = re.compile(
    r'</?(?:br|hr|p|div|span|b|i|strong|em|ul|ol|li|table|tr|td|th|h[1-6]|head|body|html|script|style|link|meta|input|button|form|label|img|a)\b[^>]*/?>', 
    flags=re.IGNORECASE
)

# Valid lowercase SVG element names — anything with an uppercase letter that isn't
# a known valid casing (SVG is case-sensitive; all standard elements are lowercase)
_VALID_SVG_TAGS = {
    'svg','g','path','rect','circle','ellipse','line','polyline','polygon',
    'text','tspan','defs','use','symbol','marker','pattern','lineargradient',
    'radialgradient','stop','clippath','mask','filter','feblend','fecolormatrix',
    'fecomponenttransfer','fecomposite','feconvolvematrix','fediffuselighting',
    'fedisplacementmap','feflood','fegaussianblur','feimage','femerge','femergenode',
    'femorphology','feoffset','fepointlight','fespotlight','fetile','feturbulence',
    'animate','animatetransform','animatemotion','set','mpath','title','desc',
    'metadata','switch','foreignobject','image','textpath','a','view','solidcolor',
}

def _strip_unknown_tags(svg_text):
    """Remove paired and self-closing tags whose name (lowercased) is not a known SVG element."""
    def _replace(m):
        tag = m.group(2).lower()
        if tag in _VALID_SVG_TAGS:
            return m.group(0)  # keep it
        return ''
    # Self-closing: <Foo ... />
    svg_text = re.sub(r'<(/?)([A-Za-z][A-Za-z0-9:_-]*)((?:\s[^>]*)?)/>', _replace, svg_text)
    # Opening/closing: <Foo ...> and </Foo>
    svg_text = re.sub(r'<(/?)([A-Za-z][A-Za-z0-9:_-]*)((?:\s[^>]*)?)>', _replace, svg_text)
    return svg_text


def repair_svg(svg_text):
    # Strip HTML-only tags that are disallowed in SVG
    svg_text = _HTML_TAG_RE.sub('', svg_text)
    # Remove any tags whose element name is not a recognised SVG element
    svg_text = _strip_unknown_tags(svg_text)
    # Escape bare ampersands
    return _BARE_AMP_RE.sub('&amp;', svg_text)


def extract_svg(text):
    m = SVG_REGEX.search(text)
    if m:
        return repair_svg(m.group(0).strip())

    # Truncation recovery
    m = SVG_OPEN_RE.search(text)
    if m:
        truncated = repair_svg(m.group(0).strip()) + "</svg>"
        try:
            root = ET.fromstring(truncated)
            if root.tag.endswith("svg"):
                return truncated
        except ET.ParseError:
            pass
    return ""


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def generate_svg(prompt, max_new_tokens=3500, debug=False):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output_ids[0, input_len:], skip_special_tokens=True)
    if debug:
        print("=== RAW MODEL OUTPUT ===")
        print(repr(decoded[:500]))
        print("========================")
    svg = extract_svg(decoded)
    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)
    return svg


def generate_svg_batch(prompts, max_new_tokens=3500):
    """
    Returns (svgs, failures). Uses true batch generation for speed.
    """
    chat_texts = [
        (
            "<|im_start|>system\n"
            f"{SYSTEM_PROMPT}<|im_end|>\n"
            "<|im_start|>user\n"
            f"{prompt}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        for prompt in prompts
    ]

    inputs = tokenizer(
        chat_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CONFIG["max_seq_length"],
    ).to(model.device)

    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded_batch = tokenizer.batch_decode(output_ids[:, input_len:], skip_special_tokens=True)
    svgs = []
    failures = []

    for prompt, decoded in zip(prompts, decoded_batch):
        svg = extract_svg(decoded)
        parse_error = None
        valid = False
        if svg:
            try:
                root = ET.fromstring(svg)
                valid = root.tag.endswith("svg")
            except ET.ParseError as e:
                parse_error = str(e)

        if not valid:
            failures.append({
                "prompt": prompt,
                "raw_decoded": decoded[:2000],
                "extracted_svg": svg[:1000] if svg else "",
                "no_svg_found": not bool(svg),
                "parse_error": parse_error,
            })
            svg = fallback_svg(prompt)
        svgs.append(svg)

    return svgs, failures


test_prompt = "a simple blue bird icon"
pred_svg = generate_svg(test_prompt, debug=True)
print(pred_svg[:500])
print("Used fallback:", pred_svg == fallback_svg(test_prompt))


/home/anthonylamelas/Coding/Deep_Learning_Midterm_1/venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


=== RAW MODEL OUTPUT ===
'<svg height="128" viewBox="-4 0 36 36" width="128" xmlns="http://www.w3.org/2000/svg"><path d="m579.4 20c-1-.2-1.9-.4-2.9-.4h-1v-1l-1 .1s-1.1 1.2-1.1 1.2z"/><circle cx="579.4" cy="20" r=".8"/></svg>'
<svg height="128" viewBox="-4 0 36 36" width="128" xmlns="http://www.w3.org/2000/svg"><path d="m579.4 20c-1-.2-1.9-.4-2.9-.4h-1v-1l-1 .1s-1.1 1.2-1.1 1.2z"/><circle cx="579.4" cy="20" r=".8"/></svg>
Used fallback: False


In [22]:
import json
import glob

# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
# TEST_PROMPTS_PATH = "/kaggle/input/svg-test-public-prompts/test_prompts.csv"
# SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Save locally
TEST_PROMPTS_PATH = TEST_CSV_PATH
_model_name = Path(trained_model_dir).name

# Resume from any existing .part file for this model, or start a fresh run
_existing_parts = sorted(glob.glob(f"./submission_{_model_name}_*.csv.part"))
if _existing_parts:
    tmp_path = _existing_parts[-1]          # most recent .part
    SUBMISSION_PATH = tmp_path[:-5]         # strip .part suffix
    print(f"[RESUME] Found existing .part file: {tmp_path}")
else:
    _run_ts = int(time.time())
    SUBMISSION_PATH = f"./submission_{_model_name}_{_run_ts}.csv"
    tmp_path = f"{SUBMISSION_PATH}.part"
    print(f"[FRESH] No existing .part file — starting from scratch: {SUBMISSION_PATH}")

DEBUG_LOG_PATH = f"./debug_{_model_name}_{int(time.time())}.jsonl"

print('Will write submission to:', SUBMISSION_PATH)
print('Will write debug log to: ', DEBUG_LOG_PATH)

test_df = pd.read_csv(TEST_PROMPTS_PATH)
total = len(test_df)

batch_size = 6
invalid_count = 0
t0 = time.time()

checkpoint_path = SUBMISSION_PATH
if os.path.exists(tmp_path):
    _part_df = pd.read_csv(tmp_path)
    done_ids = set(_part_df["id"].tolist())
    print(f"[RESUME] Loaded {len(done_ids)}/{total} already-done rows from .part file — {total - len(done_ids)} remaining")
else:
    done_ids = set()
    if os.path.exists(checkpoint_path):
        _done_df = pd.read_csv(checkpoint_path)
        done_ids = set(_done_df["id"].tolist())
        print(f"[RESUME] Final submission already exists with {len(done_ids)}/{total} rows — {total - len(done_ids)} remaining")
    else:
        print(f"[FRESH] No checkpoint found — will generate all {total} rows")

rows_buffer = []

for start in range(0, total, batch_size):
    batch_df = test_df.iloc[start:start + batch_size]
    batch_df = batch_df[~batch_df["id"].isin(done_ids)]
    if batch_df.empty:
        continue

    prompts = batch_df["prompt"].tolist()
    ids = batch_df["id"].tolist()

    svgs, failures = generate_svg_batch(prompts, max_new_tokens=3500)

    # Flush failure debug entries immediately so they survive a crash
    if failures:
        with open(DEBUG_LOG_PATH, "a") as dbf:
            for entry in failures:
                dbf.write(json.dumps(entry) + "\n")

    rows_buffer.clear()
    for sample_id, prompt, svg in zip(ids, prompts, svgs):
        if not is_valid_svg(svg):
            invalid_count += 1
            svg = fallback_svg(prompt)
        rows_buffer.append({"id": sample_id, "svg": svg})

    pd.DataFrame(rows_buffer).to_csv(
        tmp_path,
        mode="a",
        index=False,
        header=not os.path.exists(tmp_path),
    )
    done_ids.update([r["id"] for r in rows_buffer])

    if len(done_ids) % 100 == 0 or start + batch_size >= total:
        print(f"Checkpoint: {len(done_ids)}/{total} rows written")

os.replace(tmp_path, checkpoint_path)
elapsed_min = (time.time() - t0) / 60
sub_df = pd.read_csv(checkpoint_path)
print(f"Saved: {SUBMISSION_PATH}")
print(f"Debug log: {DEBUG_LOG_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()

[FRESH] No existing .part file — starting from scratch: ./submission_qwen3b_svg_lora_v2_1775092052.csv
Will write submission to: ./submission_qwen3b_svg_lora_v2_1775092052.csv
Will write debug log to:  ./debug_qwen3b_svg_lora_v2_1775092052.jsonl
[FRESH] No checkpoint found — will generate all 1000 rows


KeyboardInterrupt: 

## Notes

- Keep a fixed seed, runtime logs, and invalid-generation counts (required by `contest_docs/05`).
- If you use Kaggle-packaged datasets (`svg-train-public`, `svg-test-public-prompts`, `svg-utils`), swap paths into the loading cells.
- For stricter alignment with Unsloth templates, copy the latest prompt-formatting snippets from the official Qwen3.5-2B notebook linked above.